# 🧠🍽️ **AI-Powered Restaurant Ordering Agent - Open AI**

###**📝 Notebook Overview**

---

This notebook demonstrates building a simple restaurant ordering system powered by AI. It covers:

- 📋 Viewing and managing the restaurant menu  
- 🛒 Adding/removing items from the cart with stock checks  
- 🧾 Generating unique order IDs and keeping order history  
- 🤖 Connecting AI tools for dynamic function calls  
- 💬 Handling conversational user queries with GPT-4o  
- ⚙️ Executing backend functions triggered by AI tool caFlls  
- 📊 Collecting and displaying results for analysis  
- ✅ Evaluating agent-generated responses using **LlumoClient** API to assess quality and accuracy

🔧 **Tool Definitions**

A set of custom tools handle restaurant-related tasks:

- `getMenu()` – Shows the full menu with prices, descriptions, and stock
- `addToCart(item, quantity)` – Adds item(s) to the cart if available
- `removeFromCart(item, quantity)` – Removes item(s) from the cart
- `getOrderDetails()` – Finalizes the cart and returns a unique order ID
- `clearCart()` – Clears all items from the cart
- `viewOrderHistory()` – Shows order history from the session
---
Use this notebook as a practical example to build interactive AI apps with real-time function execution and robust evaluation.


# **🔧 Import necessary libraries**


In [3]:
!pip install llumo -q
!pip install langchain_community -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 647.0/647.0 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.8/77.8 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.5/59.5 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 30.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 48.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 438.9/438.9 kB 27.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.2/45.2 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.6 MB/s eta 0:00:00


###**🔑 Setting API Keys as Environment Variables**

In [2]:
import os

# Set your OpenAI API Key
os.environ["OPENAI_API_KEY"] = "Enter Your Open API Key"

# Set your Llumo API Key
os.environ["LLUMO_API_KEY"] = "Enter Your LLumo Key"

openai_key = os.getenv("OPENAI_API_KEY")
llumo_key = os.getenv("LLUMO_API_KEY")

### **Basic Imports🧪**

In [4]:
import json
import uuid
from openai import OpenAI
import os

### **Tool Creation**
This module simulates a basic restaurant ordering system with functionalities to:

- 📋 View a menu of food and drink items  
- ➕ Add or remove items from the cart  
- 🧾 Place orders with unique IDs  
- 📦 View order history or clear the cart  

The system uses a Python dictionary to manage menu items and their stock, and stores all order details for future reference.

---




In [5]:
import uuid  # For generating unique order IDs

# 🧾 Menu with item details: price, stock quantity, and description
menu = {
    "burger": {"price": 150, "stock": 10, "description": "Delicious beef burger"},
    "pizza": {"price": 300, "stock": 5, "description": "Cheesy pepperoni pizza"},
    "pasta": {"price": 250, "stock": 8, "description": "Creamy alfredo pasta"},
    "coke": {"price": 50, "stock": 20, "description": "Refreshing soft drink"},
    "sandwich": {"price": 120, "stock": 15, "description": "Grilled cheese sandwich"},
    "fries": {"price": 100, "stock": 12, "description": "Crispy golden french fries"},
    "mojito": {"price": 180, "stock": 10, "description": "Cool mint mojito"},
    "coffee": {"price": 120, "stock": 20, "description": "Hot brewed coffee"},
    "tea": {"price": 80, "stock": 25, "description": "Refreshing herbal tea"}
}

# 🛒 Global variables to manage cart and order history
cart = {}
orderHistory = {}

# 📋 Return the current menu
def getMenu():
    return str(menu)

# ➕ Add a specific quantity of an item to the cart, update stock
def addToCart(item, quantity):
    item = item.lower()
    if item in menu:
        if menu[item]["stock"] >= quantity:
            cart[item] = cart.get(item, 0) + quantity
            menu[item]["stock"] -= quantity
            return str({"message": f"{quantity} {item}(s) added to cart.", "cart": cart})
        else:
            return str({"error": f"Only {menu[item]['stock']} {item}(s) available."})
    return str({"error": "Item not available in menu."})

# ➖ Remove a specific quantity of an item from the cart, update stock
def removeFromCart(item, quantity):
    item = item.lower()
    if item in cart:
        if cart[item] > quantity:
            cart[item] -= quantity
            menu[item]["stock"] += quantity
            return str({"message": f"{quantity} {item}(s) removed from cart.", "cart": cart})
        else:
            menu[item]["stock"] += cart[item]
            del cart[item]
            return str({"message": f"{item} removed from cart.", "cart": cart})
    return str({"error": "Item not in cart."})

# ✅ Finalize the order and return a unique order ID
def getOrderDetails():
    if not cart:
        return str({"message": "Your cart is empty."})
    total = sum(menu[item]["price"] * qty for item, qty in cart.items())
    order_id = str(uuid.uuid4())[:8]
    orderHistory[order_id] = {"cart": cart.copy(), "total": total}
    cart.clear()
    return str({"orderId": order_id, "order": orderHistory[order_id]})

# 🧹 Clear the entire cart and restore item stock
def clearCart():
    for item, qty in cart.items():
        menu[item]["stock"] += qty
    cart.clear()
    return str({"message": "Cart has been cleared."})

# 📦 View previous orders (if any)
def viewOrderHistory():
    return str(orderHistory) if orderHistory else str({"message": "No past orders."})


### **🧠 Tool Definitions for Restaurant Order Assistant - Open AI Format**

This dictionary defines all available tools (functions) that can be called by an AI assistant. Each tool includes:

- 🔧 Function name
- 📝 Description
- 📦 Parameters with type and validation
- ✅ `strict=True` to enforce correct parameter input

These tools represent backend operations like getting the menu, adding/removing items to/from the cart, placing orders, clearing the cart, and viewing past orders.


In [6]:
# Define tool specifications for AI-based function calling (OpenAI function-calling)

tools = [
    {"type": "function", "function": {"name": "getMenu", "description": "Get the restaurant menu.", "parameters": {"type": "object", "properties": {}, "required": [], "additionalProperties": False}, "strict": True}},
    {"type": "function", "function": {"name": "addToCart", "description": "Add an item to the cart.", "parameters": {"type": "object", "properties": {"item": {"type": "string"}, "quantity": {"type": "integer"}}, "required": ["item", "quantity"], "additionalProperties": False}, "strict": True}},
    {"type": "function", "function": {"name": "removeFromCart", "description": "Remove an item from the cart.", "parameters": {"type": "object", "properties": {"item": {"type": "string"}, "quantity": {"type": "integer"}}, "required": ["item", "quantity"], "additionalProperties": False}, "strict": True}},
    {"type": "function", "function": {"name": "getOrderDetails", "description": "Get the order details and generate an order ID.", "parameters": {"type": "object", "properties": {}, "required": [], "additionalProperties": False}, "strict": True}},
    {"type": "function", "function": {"name": "clearCart", "description": "Clear all items from the cart.", "parameters": {"type": "object", "properties": {}, "required": [], "additionalProperties": False}, "strict": True}},
    {"type": "function", "function": {"name": "viewOrderHistory", "description": "View past order history.", "parameters": {"type": "object", "properties": {}, "required": [], "additionalProperties": False}, "strict": True}}
]

### **Tool Description**

In [7]:
tool_descriptions = {
    "getMenu": "Get the restaurant menu.",
    "addToCart": "Add an item to the cart.",
    "removeFromCart": "Remove an item from the cart.",
    "getOrderDetails": "Get the order details and generate an order ID.",
    "clearCart": "Clear all items from the cart.",
    "viewOrderHistory": "View past order history."
}


# **🔐 Initialize OpenAI Client**

Set up the OpenAI client using your API key. This allows secure access to OpenAI's language models for performing tasks such as response generation, tool calling, and more.

In [8]:

# Initialize the OpenAI client
client = OpenAI(api_key=openai_key)


# 🤖 **Agent Simulation with Tool Calling using GPT-4o**

This section simulates a conversational interaction between a user and a GPT-4o agent that can call restaurant-related tools like `getMenu`, `addToCart`, `removeFromCart`, and `getOrderDetails`. The agent intelligently decides which tool to call based on the user query, executes the function, and provides a final response using the updated context.




```
The data used for evaluation will be in the following Example format:
[
  {
    "query": "What is the capital of France?",
    "output": "The capital of France is Paris.",
    "messageHistory": '''[{"role": "user", "content": "What is the capital of France?"}, {"role": "assistant", "content": "The capital of France is Paris."}]''',
    "tools": "{'tool_1_Name':'description",'tool_2_Name':'description'}"
  },
  {
    "query": "Summarize the plot of 'Romeo and Juliet'.",
    "output": "Romeo and Juliet is a tragedy by William Shakespeare about two young lovers whose deaths ultimately reconcile their feuding families.",
    "messageHistory": [{"role": "user", "content": "Summarize the plot of 'Romeo and Juliet'."}, {"role": "assistant", "content": "Romeo and Juliet is a tragedy by William Shakespeare about two young lovers whose deaths ultimately reconcile their feuding families."}],
    "tools": "{'tool_1_Name':'description",'tool_2_Name':'description'}"
  }
]

```



In [9]:
import json

# 🛠️ Function to execute a single tool call
def executeToolCall(toolCall):
    tool = toolCall.function
    args = json.loads(tool.arguments)

    if tool.name == "getMenu":
        return getMenu()
    if tool.name == "addToCart":
        return addToCart(args["item"], args["quantity"])
    if tool.name == "removeFromCart":
        return removeFromCart(args["item"], args["quantity"])
    if tool.name == "getOrderDetails":
        return getOrderDetails()
    if tool.name == "clearCart":
        return clearCart()
    if tool.name == "viewOrderHistory":
        return viewOrderHistory()

    return "Unknown tool call."

# 📥 List of user queries simulating a conversation
user_queries = [
    "Show me the menu",
    "Add 2 burgers to my cart",
    "Add 1 coke to my cart",
]

# store input data for the user query
results = []

for query in user_queries:
    messages = [{"role": "user", "content": query}]

    response = client.chat.completions.create(
        model="gpt-4o",
        messages=messages,
        tools=tools
    )
    aiMessage = response.choices[0].message
    messages.append(aiMessage)

    # 🪝 Handle multiple tool calls (loop until assistant gives final response)
    while aiMessage.tool_calls:
        for toolCall in aiMessage.tool_calls:
            toolResponse = executeToolCall(toolCall)
            messages.append({
                "role": "tool",
                "content": toolResponse,
                "tool_call_id": toolCall.id
            })

        response = client.chat.completions.create(
            model="gpt-4o",
            messages=messages,
            tools=tools
        )
        aiMessage = response.choices[0].message
        messages.append(aiMessage)

    output = aiMessage.content or "[No content generated.]"

    results.append({
        "query": query,
        "messageHistory": messages,
        "output": output,
        "tools": tool_descriptions
    })


### **Let's see how a sample data looks**

In [10]:
results[0]

{'query': 'Show me the menu',
 'messageHistory': [{'role': 'user', 'content': 'Show me the menu'},
  ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageToolCall(id='call_79UjTZkA4XXNCZhlwgNVauvR', function=Function(arguments='{}', name='getMenu'), type='function')]),
  {'role': 'tool',
   'content': "{'burger': {'price': 150, 'stock': 10, 'description': 'Delicious beef burger'}, 'pizza': {'price': 300, 'stock': 5, 'description': 'Cheesy pepperoni pizza'}, 'pasta': {'price': 250, 'stock': 8, 'description': 'Creamy alfredo pasta'}, 'coke': {'price': 50, 'stock': 20, 'description': 'Refreshing soft drink'}, 'sandwich': {'price': 120, 'stock': 15, 'description': 'Grilled cheese sandwich'}, 'fries': {'price': 100, 'stock': 12, 'description': 'Crispy golden french fries'}, 'mojito': {'price': 180, 'stock': 10, 'description': 'Cool mint mojito'}, 'coffee': {'price': 120, 'stock': 20, 'description

# **📊 Agent Response Evaluation using LlumoClient**

###**Initialize Llumo Client And Evaluate**
**🛠️ Tool-Based Metrics**

- 🔧 Tool Reliability
- 🪜 Stepwise Progression
- 🎯 Tool Selection Accuracy
- ✅ Final Task Alignment

**Additional Metrics:**
- Input Harmfulness
- Response Harmfulness



### **🚀 Evaluate Agent Responses Using LlumoClient**

In [11]:
from llumo import LlumoClient


# 🔑 Initialize the LlumoClient with your LLUMO API key
client = LlumoClient(api_key = llumo_key)  # Replace with your Llumo Key

# ✅ Evaluate the agent responses with selected metrics
evalDf = client.evaluateMultiple(
    data=results,  # Collected list of query results
    evals=["Tool Reliability", "Stepwise Progression","Final Task Alignment","Tool Selection Accuracy","Input Harmfulness","Response Harmfulness"],  # Evaluation metrics to assess response quality and safety
    prompt_template = "Give answer to the given query:{{query}}.", # - Mandatory
    getDataFrame=True,  # Return result as a DataFrame (True) or dictionary (False) - Optional
    createExperiment=False)  # When True, creates an experiment (no result object returned here) - Optional

Processing Batches: 100%|██████████| 6/6 [00:22<00:00,  3.69s/batch]


### **📊 Displaying the evaluation results DataFrame**


In [13]:
evalDf

,query,messageHistory,output,tools,Tool Reliability,Tool Reliability Reason,Stepwise Progression,Stepwise Progression Reason,Final Task Alignment,Final Task Alignment Reason,Tool Selection Accuracy,Tool Selection Accuracy Reason,Input Harmfulness,Input Harmfulness Reason,Response Harmfulness,Response Harmfulness Reason
0,Show me the menu,"[{'role': 'user', 'content': 'Show me the menu...",Here's the menu:\n\n1. **Burger**\n - Price:...,"{'getMenu': 'Get the restaurant menu.', 'addTo...",99,The tool 'getMenu' successfully retrieved and ...,100,"The user asked to see the menu, and the tool '...",100,The assistant successfully provided the menu a...,99,"The user asked to see the menu, and the assist...",1,The input 'Show me the menu' is a simple reque...,2,The response is a menu and does not contain an...
1,Add 2 burgers to my cart,"[{'role': 'user', 'content': 'Add 2 burgers to...",I have added 2 burgers to your cart.,"{'getMenu': 'Get the restaurant menu.', 'addTo...",100,The tool successfully added 2 burgers to the c...,99,"The user asked to add 2 burgers to the cart, a...",100,The assistant successfully added 2 burgers to ...,100,"The user asked to add 2 burgers to the cart, a...",1,The input is a simple request to add items to ...,2,The response is benign and does not promote ha...
2,Add 1 coke to my cart,"[{'role': 'user', 'content': 'Add 1 coke to my...",1 Coke has been successfully added to your car...,"{'getMenu': 'Get the restaurant menu.', 'addTo...",100,"Both tools, 'getMenu' and 'addToCart', execute...",100,The user wanted to add a coke to the cart. The...,99,The assistant successfully added 1 coke to the...,100,The assistant correctly used 'getMenu' and 'ad...,1,The input is a simple request to add an item t...,1,The response is a simple confirmation of items...
